# 1_3 Bronze — Source Inspection & Problem-Key Discovery

Ad-hoc bronze audit toolkit (run manually — not part of the nightly job):

* **Source inventory** — live PostgreSQL `information_schema` → `crm_ingestion.bronze.source_inventory`; diffs the upstream `raw` tables against the 1_1 pipeline config (new sources, missing sources).
* **Inspect tables** — one-glance health per JSON-payload source: row count, re-dump ratio, nulls, date ranges, repair-ladder parse % on a sample, distinct keys and row shapes.
* **Discover problem keys** — `discover_all_taxonomy`: full distinct-payload scan reporting every `(key, breaker_class)` pair (apostrophe, double quote, backslash, newline, tab, smart quote, bare literal) — the evidence base for utils' `_PROBLEM_KEYS` set and the JSON repair ladder.

Connection values come from the `dea-rds` secret scope (see SETUP_secrets); shared engine functions via `%run ../utils`.

In [0]:
%run ../utils

In [0]:
# ============================================================================
# SOURCE INVENTORY — live PostgreSQL information_schema → bronze.source_inventory
# Re-run when a new source appears upstream. Works with empty bronze (queries
# PostgreSQL directly); the pipeline comparison needs 1_1's config table.
# ============================================================================
from pyspark.sql.functions import current_timestamp

SECRET_SCOPE = "dea-rds"   # same scope as 1_2_Bronze_ingestion (SETUP_secrets)
jdbc_host     = dbutils.secrets.get(scope=SECRET_SCOPE, key="host")
jdbc_port     = dbutils.secrets.get(scope=SECRET_SCOPE, key="port")
jdbc_database = dbutils.secrets.get(scope=SECRET_SCOPE, key="database")
jdbc_url      = f"jdbc:postgresql://{jdbc_host}:{jdbc_port}/{jdbc_database}"

pg_props = {
    "user":     dbutils.secrets.get(scope=SECRET_SCOPE, key="username"),
    "password": dbutils.secrets.get(scope=SECRET_SCOPE, key="password"),
    "driver":   "org.postgresql.Driver",
}

inv_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("query", """
        SELECT t.table_schema,
               t.table_name,
               COUNT(c.column_name) AS column_count
        FROM information_schema.tables t
        JOIN information_schema.columns c
          ON c.table_schema = t.table_schema
         AND c.table_name  = t.table_name
        WHERE t.table_schema = 'raw'
          AND t.table_type   = 'BASE TABLE'
        GROUP BY t.table_schema, t.table_name
        ORDER BY t.table_name
    """)
    .options(**pg_props)
    .load()
)

upstream = {r["table_name"] for r in inv_df.collect()}

# Compare against the pipeline config (1_1) — tolerates a missing config table
try:
    configured = {
        r["source_table"].split(".", 1)[1]
        for r in spark.table("crm_ingestion.config.bronze_config")
                    .filter(col("active"))
                    .collect()
    }
except Exception:
    configured = None
    print("ℹ️ config.bronze_config not present (run 1_1 first) — pipeline comparison skipped")

if configured is not None:
    inv_df = inv_df.withColumn(
        "in_pipeline", col("table_name").isin(list(configured)) if configured else lit(False)
    )
    new_sources      = sorted(upstream - configured)
    missing_upstream = sorted(configured - upstream)
    if new_sources:
        print(f"🆕 NEW upstream sources not in the pipeline: {new_sources}")
    if missing_upstream:
        print(f"⚠️ Configured sources missing upstream: {missing_upstream}")
    if not new_sources and not missing_upstream:
        print("✅ Upstream and pipeline config are in sync")
else:
    inv_df = inv_df.withColumn("in_pipeline", lit(None).cast("boolean"))

inv_df = inv_df.withColumn("inventory_date", current_timestamp())

(
    inv_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("crm_ingestion.bronze.source_inventory")
)
print(f"✅ Inventory written → crm_ingestion.bronze.source_inventory ({inv_df.count()} sources)")
display(spark.table("crm_ingestion.bronze.source_inventory"))

In [0]:
# ============================================================================
# INSPECT TABLE — health glance per bronze source
# utils has no inspect_table (only find_problem_keys + raw_lineage_tree), so
# it lives HERE: cheap, read-only, no temp tables, no writes. For the deep
# column-path view of one source, call utils' raw_lineage_tree('<name>').
# ============================================================================
from collections import Counter

_KEY_RE = re.compile(r"[\"'](\w+)[\"']\s*:")


def inspect_table(bronze_name, sample_rows=200):
    """One-glance audit of a bronze JSON-payload source. Read-only, no writes.

    Reports: raw rows, re-dump ratio (distinct md5 payload versions), null
    raw_data, date ranges, repair-ladder parse % on a random sample, and the
    distinct keys / row shapes seen in that sample. Returns a summary dict,
    or None when the table is missing, empty, or not a JSON payload source.
    """
    full = f"{CATALOG}.{BRONZE_SCHEMA}.{bronze_name}"
    try:
        df = spark.table(full)
    except Exception:
        print(f"  ❓ {bronze_name}: not found — run 1_2 first")
        return None
    if "raw_data" not in df.columns:
        print(f"  ⏭️ {bronze_name}: no raw_data column — not a JSON payload source")
        return None

    total = df.count()
    if total == 0:
        print(f"  ⏭️ {bronze_name}: 0 rows (bronze not loaded — run 1_2 first)")
        return None

    nulls    = df.filter(col("raw_data").isNull()).count()
    versions = df.select(md5(col("raw_data"))).distinct().count()
    lo, hi, upd = df.select(
        expr("min(insert_date)"), expr("max(insert_date)"), expr("max(bronze_updated_at)")
    ).collect()[0]

    frac   = min(1.0, sample_rows / total)
    sample = [
        r[0] for r in df.select("raw_data")
                        .where(col("raw_data").isNotNull())
                        .sample(False, frac)
                        .limit(sample_rows).collect()
    ]
    parsed    = sum(1 for s in sample if _parse_activity_final(s) is not None)
    parse_pct = 100.0 * parsed / len(sample) if sample else 0.0

    keys, shapes = Counter(), Counter()
    for s in sample:
        for k in set(_KEY_RE.findall(s)):
            keys[k] += 1
        try:
            shapes[type(json.loads(s.strip())).__name__] += 1
        except Exception:
            shapes["needs-repair"] += 1

    top_keys = ", ".join(f"{k}({n})" for k, n in keys.most_common(5))
    dup = f" ({total / versions:.1f}x re-dumps)" if total > versions else ""
    print(f"  🔎 {bronze_name}: {total:,} rows{dup} | nulls {nulls:,} | insert_date {lo} → {hi} | updated → {upd}")
    print(f"      parse {parse_pct:.1f}% on {len(sample)} sampled | shapes {dict(shapes)}")
    print(f"      {len(keys)} distinct keys; top: {top_keys or '—'}")
    return {
        "table": bronze_name, "rows": total, "versions": versions, "nulls": nulls,
        "parse_pct": round(parse_pct, 1), "n_keys": len(keys),
        "shapes": dict(shapes), "top_keys": [k for k, _ in keys.most_common(5)],
    }


# Loop over every bronze table (payload sources are picked up automatically;
# non-payload tables like source_inventory are skipped by the raw_data check)
bronze_tables = sorted(t.name for t in spark.catalog.listTables(f"{CATALOG}.{BRONZE_SCHEMA}"))
print(f"Inspecting {len(bronze_tables)} bronze table(s)…\n")
summaries = [r for r in (inspect_table(t) for t in bronze_tables) if r]
print(f"\nDone: {len(summaries)} JSON-payload source(s) inspected.")

In [0]:
# =============================================================================
# discover_all_taxonomy
# Full distinct-payload scan. Reports every breaker class found in any value
# of any key (including nested and stringified JSON).
# =============================================================================
import json
import re
from pyspark.sql.functions import col, md5, explode
from pyspark.sql.types import ArrayType, StringType, StructType, StructField

BREAKER_CLASSES = {
    "apostrophe":      "'",
    "dbl_quote":       '"',
    "backslash":       "\\",
    "newline":         "\n",
    "tab":             "\t",
    "smart_quote":     "“”‘’",
    "trailing_comma":  None,   # detected by structure, not by char
    "unquoted_key":    None,
    "bare_literal":    None,   # None / True / False / NaN
}

def _classify_value(value: str) -> set:
    """Return the set of breaker classes present in a single string value."""
    found = set()
    if not value or not isinstance(value, str):
        return found
    if "'" in value:                  found.add("apostrophe")
    if '"' in value:                  found.add("dbl_quote")
    if "\\" in value:                 found.add("backslash")
    if "\n" in value:                 found.add("newline")
    if "\t" in value:                 found.add("tab")
    if any(c in value for c in "“”‘’"): found.add("smart_quote")
    if re.search(r'\b(None|True|False|NaN|undefined)\b', value):
        found.add("bare_literal")
    return found

def _walk(obj, parent_key=None, results=None):
    if results is None:
        results = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            # unquoted-key style only detectable in raw text; skip here
            if isinstance(v, str):
                for cls in _classify_value(v):
                    results.add((k, cls))
                # stringified JSON — recurse
                s = v.strip()
                if s[:1] in ("{", "["):
                    try:
                        inner = json.loads(s)
                        _walk(inner, k, results)
                    except Exception:
                        pass
            else:
                _walk(v, k, results)
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, str) and parent_key:
                for cls in _classify_value(item):
                    results.add((parent_key, cls))
            else:
                _walk(item, parent_key, results)
    return results

def discover_all_taxonomy(bronze_table, payload_col="raw_data"):
    """
    Full distinct-payload scan.
    Returns every (key, breaker_class) pair found in the data.
    """
    df = (spark.table(bronze_table)
              .filter(col(payload_col).isNotNull())
              .withColumn("_ph", md5(col(payload_col)))
              .dropDuplicates(["_ph"])
              .select(payload_col))

    def _extract(raw):
        if not raw:
            return []
        try:
            parsed = json.loads(raw)
        except Exception:
            return []          # only clean JSON is walked; broken rows are separate
        pairs = _walk(parsed)
        return [{"key": k, "breaker": b} for k, b in pairs]

    schema = ArrayType(StructType([
        StructField("key", StringType()),
        StructField("breaker", StringType())
    ]))
    udf_extract = udf(_extract, schema)

    result = (df.withColumn("_pairs", udf_extract(col(payload_col)))
                .select(explode("_pairs").alias("p"))
                .select(col("p.key").alias("key"), col("p.breaker").alias("breaker"))
                .groupBy("key", "breaker")
                .count()
                .orderBy(col("count").desc()))

    print(f"Distinct payload versions scanned: {df.count():,}\n")
    rows = result.collect()
    print(f"Keys carrying breaker classes: {len({r['key'] for r in rows})}")
    for r in rows:
        print(f"  {r['key']:30}  {r['breaker']:15}  {r['count']:,}")

    return result

# =============================================================================
# RUN
# =============================================================================
tables = [
    "crm_ingestion.bronze.lead_activites_raw",
    "crm_ingestion.bronze.all_payments",
    "crm_ingestion.bronze.calendly_scheduled_events",
    "crm_ingestion.bronze.close_crm_users_raw",
    "crm_ingestion.bronze.custom_activites_raw",
    "crm_ingestion.bronze.lead_merges",
    "crm_ingestion.bronze.leads_raw",
    "crm_ingestion.bronze.mdl_users_raw",
    "crm_ingestion.bronze.student_sentiment",
]

all_results = {}
for t in tables:
    print(f"\n{'='*80}\n{t}\n{'='*80}")
    all_results[t] = discover_all_taxonomy(t)